In [12]:
import os
import sys
sys.path.insert(1, '../utils')
from cmdstanpy import CmdStanModel
from pathlib import Path

from scipy.stats import norm, entropy, skewnorm
import numpy as np
from utils import get_kl_divergence, get_true_x, list_params

In [49]:
evaluating_model='DirichletSymmetric'
transform_category='simplex'
transform='ALR'

In [51]:
# Path("stan_models").mkdir(parents=True, exist_ok=True)
# stan_filename=f'stan_models/{transform}_{evaluating_model}.stan'
# with open(stan_filename, 'w') as f:
#     f.write(f'#include ../../target_densities/{evaluating_model}.stan{os.linesep}#include ../../transforms/{transform_category}/{transform}.stan{os.linesep}')
#     f.close()
    
model = CmdStanModel(stan_file='../transforms/simplex/ALR.stan', cpp_options={"STAN_THREADS": "true"})

16:28:02 - cmdstanpy - INFO - compiling stan file /Users/meenaljhajharia/github/mjhajharia/transforms/transforms/simplex/ALR.stan to exe file /Users/meenaljhajharia/github/mjhajharia/transforms/transforms/simplex/ALR
16:28:18 - cmdstanpy - INFO - compiled model executable: /Users/meenaljhajharia/github/mjhajharia/transforms/transforms/simplex/ALR


In [52]:
# import numpy as np
# delta = np.asarray([2,1,3,4,8])
# phi = np.asarray([0.3])
# sigma =  np.asarray([1])
# t = np.cumsum(delta)
# cov =  np.power(sigma,2)/(1-np.power(phi,2))*np.power(phi, np.abs(t[:, None] - t))

matern distance

klj
gp 


In [107]:
parameters = list_params(evaluating_model='DirichletAsymmetric')
alpha = parameters[1][:10]
draws = model.sample(data = {'alpha': alpha, 'N': 10}, iter_sampling=1000, show_console=True, inits=0.5)

16:40:41 - cmdstanpy - INFO - CmdStan start processing
16:40:42 - cmdstanpy - INFO - CmdStan done processing


method = sample (Default)
sample
num_samples = 1000 (Default)
num_warmup = 1000 (Default)
save_warmup = 0 (Default)
thin = 1 (Default)
adapt
engaged = 1 (Default)
gamma = 0.050000000000000003 (Default)
delta = 0.80000000000000004 (Default)
kappa = 0.75 (Default)
t0 = 10 (Default)
init_buffer = 75 (Default)
term_buffer = 50 (Default)
window = 25 (Default)
algorithm = hmc (Default)
hmc
engine = nuts (Default)
nuts
max_depth = 10 (Default)
metric = diag_e (Default)
metric_file =  (Default)
stepsize = 1 (Default)
stepsize_jitter = 0 (Default)
num_chains = 4
id = 1 (Default)
data
file = /var/folders/c6/lxnbgsbx6_bbsdmdqxlg6ky80000gn/T/tmpq488v1nm/x2go8sub.json
init = 0.5
random
seed = 60271
output
file = /var/folders/c6/lxnbgsbx6_bbsdmdqxlg6ky80000gn/T/tmpq488v1nm/ALRuu5ey5tz/ALR-20230208164041.csv
diagnostic_file =  (Default)
refresh = 100 (Default)
sig_figs = -1 (Default)
profile_file = profile.csv (Default)
num_threads = 4 (Default)


Gradient evaluation took 3.2e-05 seconds
1000 transit

In [108]:
def get_kl_divergence(draws, n_iter):
    l = draws.draws_pd()[['x[1]']].values[:n_iter]
    rv = norm()
    r = norm.rvs(size=n_iter)
    return entropy(l, r)[0]


get_kl_divergence(draws, 1000)

0.00439778759760085

In [109]:
draws.summary()

,Mean,MCSE,StdDev,5%,50%,95%,N_Eff,N_Eff/s,R_hat
lp__,7.936570,0.048161,2.083190,4.058620,8.238000,10.764600,1870.970,7168.49,1.000810
y[1],0.006873,0.003719,0.143009,-0.226278,0.004200,0.241769,1478.380,5664.28,1.002350
y[2],0.004438,0.003714,0.142741,-0.229057,0.002072,0.244888,1476.740,5658.00,1.001550
y[3],0.001997,0.003740,0.141306,-0.230131,0.001273,0.234359,1427.310,5468.62,1.000920
y[4],0.003852,0.003662,0.142243,-0.234087,0.000991,0.239262,1508.600,5780.08,1.000950
y[5],0.002236,0.003538,0.141341,-0.228122,0.003282,0.241030,1595.990,6114.92,1.000710
y[6],0.001637,0.003616,0.141471,-0.228315,0.002628,0.233549,1530.800,5865.14,1.000700
y[7],-0.000932,0.003609,0.143334,-0.237725,-0.003042,0.234374,1577.330,6043.42,1.000770
y[8],-0.001442,0.003705,0.145357,-0.242810,-0.001486,0.233838,1538.860,5896.02,1.001390
y[9],-0.002343,0.003789,0.144748,-0.242893,-0.002793,0.233636,1459.310,5591.22,1.001340


In [110]:
import arviz as az
idata=az.from_cmdstanpy(draws)

In [111]:
from rmse import rmse

pred_x=idata.posterior.x.mean(dim=['draw','chain']).values
true_x = get_true_x(alpha=alpha, evaluating_model='DirichletAsymmetric')

rmse(true_x, pred_x)

0.000130980709042402